In [1]:
# !pip install requests
# !pip install --upgrade pip
# !pip install bs4
!pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
pd.set_option('display.max_rows', 200)

In [ ]:
url = 'https://membership.appic.org/directory/search?search=1&new_id=20524&program_type_id=1&school_name=&pdir_name=&lead_name=&appic_num=&PROG_Describe=&ID_MSI_KWORDS=&program_country_id=&APP_DUE_START=&APP_DUE_END=&POS_startdate_START=&POS_startdate_END=&ACCR_APA=&ACCR_CPA=&membership_type_id=&POS_FT_funded=&POS_PT_funded=&POS_FT_high=&POS_PT_high=&SUM_INTOTAL_1112=&STAFF_LICENSED_FT=&STAFF_LICENSED_PT=&ID_SFIP_SupervisePDs=&APPLICANT_WELCOME=APP_TYPE_SCHOOL&APP_INTERVENTION=&APP_ASSESSMENT='
# ul2 = 'https://membership.appic.org/directory/search?POS_FT_funded=&APP_DUE_START=&school_name=&POS_PT_high=&program_type_id=1&membership_type_id=&lead_name=&POS_startdate_START=&SUM_INTOTAL_1112=&search=1&APP_INTERVENTION=&ACCR_CPA=&program_country_id=&PROG_Describe=&POS_FT_high=&pdir_name=&STAFF_LICENSED_FT=&ACCR_APA=&ID_MSI_KWORDS=&APPLICANT_WELCOME=APP_TYPE_SCHOOL&APP_DUE_END=&ID_SFIP_SupervisePDs=&POS_PT_funded=&APP_ASSESSMENT=&STAFF_LICENSED_PT=&appic_num=&POS_startdate_END=&new_id=20524&&sort_by=&direction=&p=2
headers = headers = {
    "User-Agent": "Mozilla/5.0",
    # "Content-Type": "application/json"
}


r = requests.get(url, headers=headers, json = {'page': 2})
soup = BeautifulSoup(r.text, 'html.parser')

links = soup.find_all('a')

In [52]:
urls = []
for link in links:
    href = link.get('href')
    if href:
        if 'directory/display/' in href:
            urls.append(href)

In [53]:
urls[0:5]

['/directory/display/741',
 '/directory/display/183',
 '/directory/display/475',
 '/directory/display/2904',
 '/directory/display/158']

In [ ]:
base_url = 'https://membership.appic.org'
for url in urls:
    

In [2]:
# Now that we have a url, how to parse tables and info
url = 'https://membership.appic.org/directory/display/741'
# url = 'https://membership.appic.org/directory/display/376'

r = requests.get(url)
soup = BeautifulSoup(r.text, 'html.parser')

In [3]:
# Program Description
print(soup.find('div', class_ = 'program-desc').text)


Aurora Mental Health and Recovery (AMHR) provides comprehensive mental health services to people of all ages, who represent virtually all DSM diagnostic categories. Services include outpatient and intensive treatment for early childhood, school-age children, adolescents, adults, and older adults. The psychology internship program allows interns to select and match to one of three tracks: child and family focused; adult focused; or placement within the immigrant and refugee track. The internship program will be accepting four interns this coming year (1- child track; 2 - adult track; 1 - immigrant and refugee track).
As part of the match process to the tracks, interns are placed on a year long primary placement with either Adult, Child and Family, or Immigrant and Refugee emphasis. In addition to the primary placement, each intern completes two minor rotations during the year that can be with any age group regardless of track assignment. The primary placement is 16 hours per week (i.e.

In [4]:
df = pd.DataFrame()
tables = soup.find_all('table')

for table in tables[:18]:
    captions = table.find_all('caption')
    if captions:
        caption = captions[0].text
        fields = [f.text.strip() for f in table.find_all('th', scope = 'row')]
        values = [f.text.strip() for f in table.find_all('td')]
        for f,v in zip(fields, values):
            df[f'{caption}_{f}'] = [v]
    else:
        fields = [f.text.strip() for f in table.find_all('th', scope = 'row')]
        values = [f.text.strip() for f in table.find_all('td')]
        for f,v in zip(fields, values):
            df[f] = [v]

df['Program Description'] = soup.find('div', class_ = 'program-desc').text.strip()

In [5]:
df.shape

(1, 55)

In [172]:
for i, table in enumerate(tables[:18]):

    for row in table.find_all('tr'):
        th = row.find('th')
        td = row.find('td')
        if td and th and (th.get('scope') == 'row' or th.get('scope') is None):
            print(th.text.strip(), td.text.strip())
            print('-'*50)

    print(i, '+'*100)

APPIC Member Number: 1692
--------------------------------------------------
Program Type: Internship
--------------------------------------------------
Membership Type: Full Membership
--------------------------------------------------
Site: Aurora Mental Health and Recovery
--------------------------------------------------
Department: Psychology Internship Program
--------------------------------------------------
Address: 1290 Chambers Road 
            
            
            
            Aurora, Colorado 80011
--------------------------------------------------
Country: United States
--------------------------------------------------
Metro Area: Not Applicable
             
             Denver-Aurora-Broomfield, CO
--------------------------------------------------
Distance from Major City: 
--------------------------------------------------
Phone: 303-617-2300
--------------------------------------------------
Fax: 
--------------------------------------------------
Training Di

In [166]:
table = tables[11]
for row in table.find_all('tr'):
    print(row)
    print('-'*50)

<tr>
<th scope="row">Accepting Applicants:</th>
<td>Yes </td>
</tr>
--------------------------------------------------
<tr>
<th scope="row">Application Due Date:</th>
<td>11/01/2025 11:59 PM EST</td>
</tr>
--------------------------------------------------
<tr>
<th scope="row">Interviews at this site are:</th>
<td>
                Required
            </td>
</tr>
--------------------------------------------------
<tr>
<th scope="row">Interview notification date:</th>
<td>12/15/2025</td>
</tr>
--------------------------------------------------
<tr>
<th scope="row">Tentative interview date:</th>
<td>01/02/2026</td>
</tr>
--------------------------------------------------
<tr>
<th scope="row">Interview process description:</th>
<td> <p>We offer virtual interviews during the first three weeks of January, possibly in December 2025 and some later dates in January, 2026. </p>
<p>Applicants will be notified via email NO LATER THAN December 15, 2025 as to whether they will be offered an intervi

In [175]:
table = tables[11]
table.find_all('tr')[2]
# for row in table.find_all('tr'):
#     # print(row)
#     # print('-'*50)
#     th = row.find('th')
#     td = row.find('td')
#     if td and th and (th.get('scope') == 'row' or th.get('scope') is None):
#         print(th.text.strip(), td.text.strip())
#         # print(th, td.text.strip())
#         print('-'*50)

<tr>
<th scope="row">A Virtual Interview are:</th>
<td> Required  
      
      <tr>
<th scope="row">Interview notification date:</th>
<td>12/05/2025</td>
</tr>
<tr>
<th scope="row">Tentative interview date:</th>
<td>01/07/26, 01/08/26, 01/13/26, 01/14/26</td>
</tr>
<tr>
<th scope="row">Interview process description:</th>
<td> <p><span style="font-size: 14px;">Based on the quality of the application and the goodness of fit between the applicant’s training goals and the internship program, approximately forty applicants are invited to interview. We interview approximately 20 Adult Track, 10 Child/Family Track, and 10 International Immigrant and Refugee Center track applicants. </span></p>
<p><span style="font-size: 12px;"><strong>All parts of the interview day will be held via video conference and interview days are scheduled for January 7th, 8th, 13th and 14th.</strong> Interview days run 9:00am to 3:00pm (MST zone) and include time with the Psychology Training Committee, the current i